# BSAN391 - Wozac - Linear Approach - Gurobi

This notebook shows how to implement and solve the Wozac planning problem using a Linear Programming approach using Python/Gurobi.

The algebraic formulation corresponding to Wozac's LP is presented below. Decision variables ${\color{red}y_{t}}$ represent the number of units to produce in period $t$, while decision variable ${\color{red}x}$ represents the capacity to build.
(Note that periods are numbered from 0 to $N-1$ to match Python's numbering from 0)

\begin{equation*}
\begin{aligned}
& {\text{maximize}}
& & \sum_{t=0}^{N-1}(s-p){\color{red}y_{t}} - {\color{red}x}(b+Nc) & \color{blue}{\longrightarrow \textbf{Profit maximization (1)}}\\[2mm]
& \text{subject to}&&\\[2mm]
& & & {\color{red}y_{t}}  \leq  dg^{t}, \text{ for } t=0, \ldots , N-1 & \color{blue}{\longrightarrow \textbf{Each year's production $\leq$ demand (2)}}\\[2mm]
& & & 0 \leq {\color{red}y_{t}}  \leq  {\color{red}x}, \text{ for } t=0, \ldots , N-1 & \color{blue}{\longrightarrow \textbf{Each year's production $\leq$ capacity to build (3)}}\\[2mm]
& & & {\color{red}x}  \geq  0\\[2mm]
\end{aligned}
\end{equation*}

In [1]:
# Libraries imports
from gurobipy import *

In [2]:
# Parameters
d = 50000       # Current Demand
g = 1.05        # Annual growth in demand
b = 16          # Unit building capacity cost
s = 3           # Unit selling price
p = 0.2         # Unit production cost
c = 0.4         # Unit capacity operating cost
N = 10          # Number of time periods

In [3]:
# Problem object definition
wozac = Model("Wozac")

Academic license - for non-commercial use only - expires 2022-08-04
Using license file C:\Users\novoadlj\gurobi.lic


In [4]:
# Decision Variables (note that nonnegativity constraints are defined here - last 0)
y = wozac.addVars(list(range(N)), name = "Production")
x = wozac.addVar(name="Capacity")

In [5]:
# First set of constraints (see (2) in the algebraic formulation above)
production1 = wozac.addConstrs(y[t] <= d*(g**t) for t in range(N))

In [6]:
# Second set of constraints (see (3) in the algebraic formulation above)
production2 = wozac.addConstrs(y[t] <= x for t in range(N))

In [7]:
# Objective Function
wozac.setObjective(sum((s-p)*y[t] for t in range(N)) - x*(b + N*c), GRB.MAXIMIZE)

In [8]:
# Run optimization engine
wozac.optimize()

Gurobi Optimizer version 9.1.2 build v9.1.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 20 rows, 11 columns and 30 nonzeros
Model fingerprint: 0x5dd12a3b
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [3e+00, 2e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+04, 8e+04]
Presolve removed 10 rows and 0 columns
Presolve time: 0.01s
Presolved: 10 rows, 11 columns, 20 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.9288176e+05   1.201423e+04   0.000000e+00      0s
       5    4.1930000e+05   0.000000e+00   0.000000e+00      0s

Solved in 5 iterations and 0.03 seconds
Optimal objective  4.193000000e+05


In [9]:
# Display optimal production plan
for v in wozac.getVars():
    print(v.varName, v.x)
    
print("Optimal total profit:", "$"+str(wozac.objVal))

Production[0] 50000.0
Production[1] 52500.0
Production[2] 55125.0
Production[3] 55125.0
Production[4] 55125.0
Production[5] 55125.0
Production[6] 55125.0
Production[7] 55125.0
Production[8] 55125.0
Production[9] 55125.0
Capacity 55125.0
Optimal total profit: $419300.0
